# End-to-End Vertex AI Batch Inference with the `google-genai` SDK

A fully automated pipeline that prepares data, uploads it to GCS, submits a Gemini batch job, polls until completion, and retrieves + parses the results.

> **Why batch inference?** For large, latency-tolerant workloads (offline enrichment, dataset labeling, bulk summarization), batch mode is **~50% cheaper** than online calls and sidesteps per-request rate limits. You hand Vertex a file of requests, it processes them asynchronously, and writes a file of responses back to GCS.

**Workflow:** `prep → upload → submit → poll → retrieve → parse`

## 1. Requirements

Run the cell below once per environment. The unified `google-genai` SDK uses the *same* client for both the Gemini Developer API and Vertex AI &mdash; we use the Vertex backend here.

In [ ]:
# The new *unified* Google GenAI SDK — one client talks to both the
# Gemini Developer API and Vertex AI. We use the Vertex backend below.
%pip install --upgrade google-genai

# Used to upload the input file and download/parse the output files.
%pip install --upgrade google-cloud-storage

### Prerequisites checklist

| Requirement | How to satisfy it |
|---|---|
| **Google Cloud Project** with billing enabled | `gcloud config set project YOUR_PROJECT_ID` |
| **Vertex AI API enabled** | `gcloud services enable aiplatform.googleapis.com` |
| **A GCS bucket** in the same region you'll run in | `gcloud storage buckets create gs://YOUR_BUCKET --location=us-central1` |
| **Application Default Credentials (ADC)** | `gcloud auth application-default login` |
| **IAM roles** on your user/service account | `roles/aiplatform.user` + `roles/storage.objectAdmin` |

> **Why ADC instead of API keys?** On the Vertex backend the SDK authenticates via Google Cloud IAM, not API keys. ADC lets the *same code* run unchanged on your laptop (your user creds) and in production (a service account) &mdash; the SDK just picks up whatever credentials the environment provides. No secrets in code.

Run this once in a terminal before executing the notebook:

```bash
gcloud auth application-default login
```

## 2. Imports & Configuration

Centralised config so the pipeline is environment-driven. In production, swap these literals for `os.environ` lookups so the same notebook promotes cleanly across dev/stage/prod.

In [ ]:
import json
import time
import datetime
from pathlib import Path

from google import genai
from google.genai.types import CreateBatchJobConfig, JobState
from google.cloud import storage

# ----------------------------------------------------------------------------
# CONFIGURATION — edit these for your environment.
# ----------------------------------------------------------------------------
PROJECT_ID = "your-gcp-project-id"
LOCATION = "us-central1"            # Must be a region where batch + the model are available.
BUCKET_NAME = "your-gcs-bucket"     # Bucket only — no gs:// prefix here.
MODEL = "gemini-2.5-flash"          # Fast/cheap; swap for gemini-2.5-pro if you need more reasoning.

# GCS "folder" prefixes. GCS has no real folders, just object name prefixes,
# but organising input/ and output/ keeps the bucket navigable and makes
# lifecycle rules (auto-delete old runs) trivial to target.
INPUT_BLOB = "batch-demo/input/input.jsonl"
OUTPUT_PREFIX = "batch-demo/output/"

LOCAL_INPUT_FILE = "input.jsonl"
LOCAL_OUTPUT_DIR = Path("./batch_results")

POLL_INTERVAL_SECONDS = 60          # Batch jobs run for minutes-to-hours; polling faster just wastes API calls.

## Step 1 &mdash; Data Prep

Write a JSONL file where each **line** is one independent inference request. The schema (`contents[].role` + `contents[].parts[].text`) is identical to a live `generate_content` call &mdash; so a prompt you validated interactively behaves identically in batch. That parity is the point: prototype online, scale offline.

In [ ]:
def create_input_jsonl(path: str) -> None:
    """Write a JSONL file where each LINE is one independent inference request.

    Vertex batch expects each line wrapped in a top-level "request" key whose
    value is a standard GenerateContentRequest.
    """
    sample_prompts = [
        "Explain quantum entanglement to a 10-year-old in three sentences.",
        "Write a haiku about distributed systems.",
        "List three reasons batch inference is cheaper than online inference.",
    ]

    # JSONL is NOT a JSON array — it's one self-contained JSON object per line.
    # This streaming format lets Vertex process millions of rows without ever
    # loading the whole file into memory.
    with open(path, "w", encoding="utf-8") as f:
        for prompt in sample_prompts:
            request = {
                "request": {
                    "contents": [
                        {
                            # "user" marks the human turn. For few-shot/multi-turn
                            # context you'd append more entries (alternating
                            # user/model roles) to this list.
                            "role": "user",
                            "parts": [{"text": prompt}],
                        }
                    ]
                    # Optional per-request params, e.g.
                    # "generationConfig": {"temperature": 0.2, "maxOutputTokens": 256}
                }
            }
            # ensure_ascii=False keeps Unicode (emoji, accents) intact rather
            # than bloating the file with \uXXXX escapes.
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"[prep] Wrote {len(sample_prompts)} requests -> {path}")


create_input_jsonl(LOCAL_INPUT_FILE)

## Step 2 &mdash; GCS Upload

Batch jobs read input from GCS (not local disk) because the job runs on Google's managed infrastructure, asynchronously, long after this notebook cell finishes. GCS is the durable hand-off point between you and Vertex.

In [ ]:
def upload_to_gcs(local_path: str, bucket_name: str, blob_name: str) -> str:
    """Upload the local JSONL to GCS and return its gs:// URI."""
    # The storage client auto-discovers the same ADC credentials as the genai
    # client — one auth story for the whole pipeline.
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    blob.upload_from_filename(local_path)

    gcs_uri = f"gs://{bucket_name}/{blob_name}"
    print(f"[upload] {local_path} -> {gcs_uri}")
    return gcs_uri


input_uri = upload_to_gcs(LOCAL_INPUT_FILE, BUCKET_NAME, INPUT_BLOB)
output_uri = f"gs://{BUCKET_NAME}/{OUTPUT_PREFIX}"

## Step 3 &mdash; Job Submission

Submit returns *immediately* with a job handle &mdash; it does not block. Decoupling submit from wait means an orchestrator can fire many jobs and reconcile them later, instead of holding a thread per job.

`vertexai=True` flips the SDK to the Vertex backend (IAM auth, GCS I/O, regional endpoints) instead of the API-key-based Developer API.

In [ ]:
def submit_batch_job(input_uri: str, output_uri: str):
    """Submit the batch job and return (job, client). Does NOT block."""
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

    job = client.batches.create(
        model=MODEL,
        # src is the GCS input. (It also accepts a BigQuery table URI if your
        # data already lives in BQ — same call, different source.)
        src=input_uri,
        config=CreateBatchJobConfig(
            # dest is a PREFIX/folder, not a filename. Vertex creates a unique
            # subfolder under it per run so concurrent jobs never collide.
            dest=output_uri,
        ),
    )

    print(f"[submit] Job created: {job.name}")
    print(f"[submit] Initial state: {job.state}")
    return job, client


job, client = submit_batch_job(input_uri, output_uri)

## Step 4 &mdash; Polling / Monitoring

We must re-`GET` the job each loop: the original handle is a point-in-time snapshot and never mutates on its own. Polling the live resource is how we observe real progress. The loop exits only on a **terminal** state, which can never change again.

In [ ]:
def poll_until_complete(client, job_name: str):
    """Block, re-fetching job state every POLL_INTERVAL_SECONDS until terminal."""
    terminal_states = {
        JobState.JOB_STATE_SUCCEEDED,
        JobState.JOB_STATE_FAILED,
        JobState.JOB_STATE_CANCELLED,
        JobState.JOB_STATE_PAUSED,
    }

    while True:
        job = client.batches.get(name=job_name)
        now = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[poll {now}] state = {job.state}")

        if job.state in terminal_states:
            return job

        # Sleep between polls so we don't hammer the API or burn quota while
        # the job churns through (potentially) millions of rows.
        time.sleep(POLL_INTERVAL_SECONDS)


final_job = poll_until_complete(client, job.name)

## Step 5 &mdash; Output Retrieval + Parse

Vertex writes results into an auto-generated subfolder under our `dest` prefix (e.g. `.../output/prediction-model-2024-.../predictions.jsonl`). We read the job's reported output location rather than guessing the path &mdash; the SDK tells us exactly where it wrote. Large outputs may be **sharded** across multiple files, so we iterate.

In [ ]:
def retrieve_and_parse(job, bucket_name: str, local_dir: Path) -> None:
    """Download every output JSONL Vertex produced and print each response."""
    local_dir.mkdir(parents=True, exist_ok=True)

    # Ask the job where it actually wrote output instead of reconstructing the
    # path ourselves; Vertex appends a unique run folder we can't predict.
    output_loc = job.dest.gcs_uri  # e.g. "gs://bucket/batch-demo/output/prediction-..."
    print(f"[retrieve] Output location: {output_loc}")

    # Strip the gs://bucket/ prefix to get the object-name prefix to list under.
    prefix = output_loc.replace(f"gs://{bucket_name}/", "")

    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)

    # Vertex may shard large outputs into multiple files, so iterate rather
    # than assume a single file.
    blobs = [b for b in bucket.list_blobs(prefix=prefix) if b.name.endswith(".jsonl")]
    if not blobs:
        print("[retrieve] No .jsonl output found — check the job for errors.")
        return

    for blob in blobs:
        local_path = local_dir / Path(blob.name).name
        blob.download_to_filename(local_path)
        print(f"[retrieve] Downloaded {blob.name} -> {local_path}")

        # Each output line mirrors the input line but adds a "response" field
        # (or an error field if that row failed). Per-row errors are why batch
        # never raises for one bad prompt — you must inspect each record.
        with open(local_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f, start=1):
                record = json.loads(line)

                if "response" in record:
                    try:
                        text = record["response"]["candidates"][0]["content"]["parts"][0]["text"]
                    except (KeyError, IndexError):
                        # Safety blocks / empty candidates land here — surface
                        # the raw record so nothing fails silently.
                        text = f"<no text — raw: {json.dumps(record['response'])[:200]}>"
                    print(f"\n--- Result {i} ---\n{text}")
                else:
                    # Row-level failure (bad prompt, safety block, quota, etc.).
                    print(f"\n--- Result {i} (ERROR) ---\n{json.dumps(record, indent=2)}")

## Run / Finalize

Retrieve only on success; otherwise surface the failure reason (`job.error` carries a human-readable message).

In [ ]:
if final_job.state == JobState.JOB_STATE_SUCCEEDED:
    print("\n[pipeline] Job SUCCEEDED — retrieving results.")
    retrieve_and_parse(final_job, BUCKET_NAME, LOCAL_OUTPUT_DIR)
else:
    print(f"\n[pipeline] Job ended in {final_job.state}.")
    if getattr(final_job, "error", None):
        print(f"[pipeline] Error: {final_job.error}")

---
## Appendix &mdash; One-shot orchestration

The cells above run each stage interactively (ideal for debugging). For an automated/scheduled run, the single function below wires all five stages into one linear call you can drop into Airflow, a Cloud Run job, or `/colab-ready`.

In [ ]:
def run_pipeline() -> None:
    """Wire the five stages together into one linear, automatable run."""
    create_input_jsonl(LOCAL_INPUT_FILE)                                  # 1. Prep
    in_uri = upload_to_gcs(LOCAL_INPUT_FILE, BUCKET_NAME, INPUT_BLOB)     # 2. Upload
    out_uri = f"gs://{BUCKET_NAME}/{OUTPUT_PREFIX}"
    j, cl = submit_batch_job(in_uri, out_uri)                            # 3. Submit
    done = poll_until_complete(cl, j.name)                               # 4. Poll

    if done.state == JobState.JOB_STATE_SUCCEEDED:                       # 5. Retrieve
        print("\n[pipeline] Job SUCCEEDED — retrieving results.")
        retrieve_and_parse(done, BUCKET_NAME, LOCAL_OUTPUT_DIR)
    else:
        print(f"\n[pipeline] Job ended in {done.state}.")
        if getattr(done, "error", None):
            print(f"[pipeline] Error: {done.error}")


# Uncomment to run the whole pipeline end-to-end in one shot:
# run_pipeline()

---
## Schema Reference

**Input line** (what you write):
```json
{"request": {"contents": [{"role": "user", "parts": [{"text": "your prompt"}]}]}}
```

**Output line** (what Vertex writes back) &mdash; input is echoed, `response` is appended:
```json
{
  "request":  { "contents": [ ... ] },
  "response": { "candidates": [ { "content": { "parts": [ { "text": "..." } ] } } ] }
}
```

## Production Hardening Notes

- **Idempotency / collisions:** keep `dest` as a stable prefix &mdash; Vertex's per-run subfolder already prevents output clashes. For inputs, suffix the blob with a timestamp/run-ID.
- **Resilience:** wrap `client.batches.get` in a retry-with-backoff for transient 5xx; the poll loop already survives indefinitely-long jobs.
- **Cost control:** batch is ~50% of online price, but set a GCS **lifecycle rule** to auto-purge old `output/` runs.
- **Fan-out:** since submit and poll are decoupled, an orchestrator can persist `job.name` to a DB, exit, and reconcile jobs in a separate scheduled task instead of holding the process open.
- **Per-row failures:** a job can be `SUCCEEDED` while individual rows errored &mdash; always inspect each output record (the parser already branches on this).